# Week 2 — Lab
## Projecting real text and image embedding spaces

In this lab we work with two pre-computed embedding sets:

- **Text:** 1500 documents from 6 categories of 20-newsgroups, embedded with the
  `all-MiniLM-L6-v2` sentence transformer (384-d).
- **Images:** 2000 images from CIFAR-10, embedded with the penultimate layer of a
  pretrained ResNet-18 (512-d).

Both are generated by `data/download.py`. The lab itself is GPU-free and runs in a few
minutes on a laptop.

We will:

1. Project each embedding space with PCA, t-SNE, and UMAP and compare.
2. Run a small **hyperparameter sweep** for UMAP on the text embeddings and pick a
   setting based on a quantitative score plus the visual.
3. Build an **interactive 3D Plotly view** of the image embeddings, with hover text
   that links each point to its CIFAR-10 class.
4. Quantify what each projection preserves with trustworthiness and continuity.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness
import umap

plt.style.use("../../assets/mplstyle/course.mplstyle")
RNG = np.random.default_rng(0)

text = np.load("../data/text_embeddings.npz", allow_pickle=True)
img  = np.load("../data/image_embeddings.npz", allow_pickle=True)

X_text, y_text = text["embeddings"], text["labels"]
names_text = list(text["label_names"])
X_img, y_img = img["embeddings"], img["labels"]
names_img = list(img["label_names"])

print(f"text:   {X_text.shape}   {len(names_text)} classes")
print(f"images: {X_img.shape}   {len(names_img)} classes")


## 1. Three projections, side by side

Run PCA, t-SNE, and UMAP on the text embeddings and look at all three at once. PCA is a
useful baseline — when it already separates classes, the more expensive nonlinear
methods are not buying you anything.

In [ ]:
def project_all(X, random_state=0):
    pca   = PCA(n_components=2, random_state=random_state).fit_transform(X)
    tsne  = TSNE(n_components=2, perplexity=30, init="pca",
                 learning_rate="auto", random_state=random_state).fit_transform(X)
    ump   = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                      random_state=random_state).fit_transform(X)
    return pca, tsne, ump


pca_t, tsne_t, umap_t = project_all(X_text)


In [ ]:
def plot_three(projs, y, names, suptitle):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.4))
    for ax, emb, title in zip(axes, projs, ["PCA", "t-SNE", "UMAP"]):
        for c, name in enumerate(names):
            m = y == c
            ax.scatter(emb[m, 0], emb[m, 1], s=8, alpha=0.8, label=name)
        ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
        ax.set_aspect("equal")
    axes[-1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    fig.suptitle(suptitle, fontsize=12)
    plt.tight_layout()
    plt.show()


plot_three([pca_t, tsne_t, umap_t], y_text, names_text,
           "Newsgroup embeddings — three projections")


PCA shows that the 6 newsgroup classes are partly linearly separable — the topic
categories (sports vs. politics vs. computing) are roughly spread along the first two
principal directions. t-SNE and UMAP both sharpen the separation, with UMAP producing
the cleanest "topic islands".

For the image embeddings, the picture is different — ResNet-18 features have much
heavier overlap between visually similar classes (cats vs. dogs, deer vs. horse).

In [ ]:
pca_i, tsne_i, umap_i = project_all(X_img, random_state=0)
plot_three([pca_i, tsne_i, umap_i], y_img, names_img,
           "CIFAR-10 ResNet-18 embeddings — three projections")


## 2. UMAP hyperparameter sweep

The two hyperparameters that matter for UMAP are `n_neighbors` (local vs. global focus)
and `min_dist` (how tightly clusters pack). Sweep them on the text data and use
trustworthiness as a quantitative cross-check.

In [ ]:
def umap_sweep(X, n_neighbors_list, min_dist_list, random_state=0):
    results = []
    embeds = {}
    for nn in n_neighbors_list:
        for md_ in min_dist_list:
            emb = umap.UMAP(n_neighbors=nn, min_dist=md_,
                            random_state=random_state).fit_transform(X)
            tw = trustworthiness(X, emb, n_neighbors=10)
            results.append(dict(n_neighbors=nn, min_dist=md_, trustworthiness=tw))
            embeds[(nn, md_)] = emb
    return pd.DataFrame(results), embeds


grid_nn = [5, 15, 50, 200]
grid_md = [0.0, 0.1, 0.5]
results, embeds = umap_sweep(X_text, grid_nn, grid_md)
results.sort_values("trustworthiness", ascending=False).head()


In [ ]:
# Plot the sweep grid
fig, axes = plt.subplots(len(grid_md), len(grid_nn),
                         figsize=(13, 9), sharex=False, sharey=False)
for j, md_ in enumerate(grid_md):
    for i, nn in enumerate(grid_nn):
        emb = embeds[(nn, md_)]
        ax = axes[j, i]
        ax.scatter(emb[:, 0], emb[:, 1], c=y_text, cmap="tab10", s=4, alpha=0.7)
        ax.set_xticks([]); ax.set_yticks([])
        tw = results.query("n_neighbors == @nn and min_dist == @md_")["trustworthiness"].iloc[0]
        ax.set_title(f"nn={nn}, md={md_}  (T={tw:.3f})", fontsize=10)
plt.tight_layout()
plt.show()


**Reading the sweep.**

- Trustworthiness is high across the grid (the underlying classes really are separable).
- Visually, `n_neighbors=15, min_dist=0.1` gives the most readable layout for this
  dataset — tight enough to count classes, not so tight that it implies false structure.
- Notice how `min_dist=0.0` produces tightly packed islands but exaggerates inter-cluster
  gaps — a reader who treats those gaps as distances will be misled.

Trustworthiness is **necessary but not sufficient**. A sweep where every setting scores
0.99 is not telling you which to pick; the visual is the tiebreaker.

## 3. Interactive 3D Plotly view

A 3D scatter is genuinely useful when there is structure that 2D flattens. Plotly's
`scatter_3d` gives you rotation, zoom, and hover-text out of the box.

In [ ]:
umap_3d = umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.1,
                    random_state=0).fit_transform(X_img)

df_img = pd.DataFrame(umap_3d, columns=["x", "y", "z"])
df_img["label"] = [names_img[c] for c in y_img]

fig = px.scatter_3d(df_img, x="x", y="y", z="z", color="label",
                    opacity=0.7,
                    title="CIFAR-10 ResNet-18 features — UMAP 3D",
                    height=600)
fig.update_traces(marker=dict(size=3))
fig.update_layout(legend=dict(itemsizing="constant"))
fig.show()


**What to look for in the rotation.**

- The "vehicle" classes (`airplane`, `automobile`, `ship`, `truck`) form a loose
  super-cluster.
- The "animal" classes form a separate one.
- Within each, fine-grained classes overlap in the directions where ResNet-18 features
  are similar — cat/dog and deer/horse are notoriously confused by small CNNs trained
  on ImageNet and finetuned on CIFAR.

A 3D view shows the super-cluster structure that 2D often flattens.

## 4. Quantify what's preserved

Trustworthiness and continuity together give a more complete picture than either
alone.

In [ ]:
def continuity(X, emb, n_neighbors=10):
    # sklearn doesn't ship continuity directly, but it is just trustworthiness with
    # the roles of X and emb swapped.
    return trustworthiness(emb, X, n_neighbors=n_neighbors)


rows = []
for name, proj in [("PCA", pca_t), ("t-SNE", tsne_t), ("UMAP", umap_t)]:
    rows.append(dict(
        method=name,
        trustworthiness=trustworthiness(X_text, proj, n_neighbors=10),
        continuity=continuity(X_text, proj, n_neighbors=10),
    ))
pd.DataFrame(rows).round(3)


**Reading the table.**

- High trustworthiness, low continuity: the projection puts genuine neighbours together
  but also smashes far-apart points together (false neighbours in the embedding).
- Low trustworthiness, high continuity: the projection scatters genuine neighbours but
  doesn't invent fake ones.
- Both high: you are in the best place. PCA usually beats t-SNE on continuity and loses
  on trustworthiness; UMAP is typically balanced.

Reporting both numbers next to your scatter is the difference between "look at my nice
clusters" and "here's what the projection preserves and what it doesn't".

## What to do differently in your own research

- Run **at least two** projection methods (PCA + one nonlinear). When they agree on the
  qualitative structure, you can trust it more. When they disagree, neither is "right"
  — the embedding lives in high dimensions and you are sampling slices.
- **Sweep hyperparameters and report the grid**, at minimum in an appendix.
- **Quote trustworthiness / continuity** when you publish an embedding scatter.
- **Resist the temptation to over-interpret cluster geometry.** Distances and density
  in 2D are not what they look like.

### Next

Try `exercises/01-umap-sweep.ipynb` for hands-on hyperparameter exploration on a held-out
embedding.